# Landmark Chronos Feature Extraction

Generate Chronos embedding features over the landmark feature table (`general_db.csv`, produced by `new_analysis.ipynb`) and persist them as `general_db_lm_chronos.csv` (consumed by `landmarks/scripts/grid_search.py`). Each (label, subject, rep) sequence is reduced to a single feature vector by averaging Chronos encoder outputs across time and feature axes.

## Configuration

In [ ]:
INPUT_CSV = "general_db.csv"
OUTPUT_CSV = "general_db_lm_chronos.csv"
CHRONOS_MODEL = "amazon/chronos-t5-small"


## Setup

In [ ]:
import pandas as pd
import numpy as np
import os, glob
from tqdm import tqdm


## Load Chronos pipeline and source data

In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().resolve().parents[1]))  # repo root, run from <modality>/notebooks
from common.chronos_features import embed_dataframe, load_pipeline

pipeline = load_pipeline(CHRONOS_MODEL)

df = pd.read_csv(INPUT_CSV)
print(df.shape)


## Sanity-check one (label, subject, rep) slice

In [ ]:
df[(df.label == 0) & (df.subject == 0) & (df.rep == 0)]

## Compute embeddings per (label, subject, rep)

In [ ]:
# One embedding per execution; each channel is embedded as its own series over time.
new_db = embed_dataframe(pipeline, df, group_cols=["label", "subject", "rep"])
assert not new_db.drop(columns=["label", "subject", "rep"]).duplicated().any(), "identical embeddings across executions"


## Aggregate into a DataFrame

In [ ]:
new_db

## Save and inspect

In [ ]:
new_db.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

In [ ]:
new_db["label"].value_counts()*0.8